# Curs 1 — Primul apel către un LLM
## LLM-uri, API-uri și parametri de generare
În acest notebook facem primul apel către un model Gemini folosind Python.
Scopul nu este să construim încă un agent, ci să înțelegem structura minimă a unui apel către un LLM:
- modelul folosit
- mesajele trimise către model
- parametrii de generare
- outputul primit
La finalul notebook-ului trebuie să avem:
1. API key configurată corect în `.env`
2. un prim apel funcțional către Gemini
3. un rezumat neutru generat de model
4. un mic experiment cu `temperature`
5. un output salvat împreună cu parametrii folosiți
---


## 0. Instalare pachete

In [2]:
# Rulează o singură dată. Dacă ești în Google Colab, sari peste acest pas.
%pip install -q openai python-dotenv requests


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install google-genai

  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
   ---------------------------------------- 0.0/790.4 kB ? eta -:--:--
   -------------------------- ------------- 524.3/790.4 kB 4.4 MB/s eta 0:00:01
   ---------------------------------------- 790.4/790.4 kB 3.9 MB/s  0:00:00
Using cached tenacity-9.1.4-py3-none-any.whl (28 kB)
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -------- ------------------------------- 0.8/3.8 MB 5.0 MB/s eta 0:00:01
   ------------------- -------------------- 1.8/3.8 MB 4.9 MB/s eta 0:00:01
   --------------------------------- ------ 3.1/3.8 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 4.9 MB/s  0:00:00

   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------- 0/7 [websockets]
   ---------------------------------------- 0/7 [websockets]
   ------------------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Instalare biblioteci
Folosim biblioteca `openai`, dar nu folosim un model OpenAI.
Folosim Gemini prin endpoint-ul compatibil cu formatul OpenAI.
-  sintaxa `messages = [{"role": "...", "content": "..."}]` este foarte răspândită și ne va ajuta mai târziu să lucrăm și cu OpenRouter, DeepSeek sau alte modele compatibile.

## 2. Configurarea cheii API

https://aistudio.google.com/api-keys

Cheia API nu se scrie direct în notebook.
O salvăm într-un fișier local numit `.env`.
Fișierul `.env` trebuie să conțină:
```text
GEMINI_API_KEY=cheia_ta_aici

!Important:

nu urca niciodată .env pe GitHub
în repository trebuie să existe doar .env.example
dacă ai publicat cheia accidental, șterge cheia din Google AI Studio și generează alta

In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from google import genai


## 3. Creăm clientul API
Clientul este obiectul prin care trimitem cereri către model.
Aici avem trei lucruri importante:
- `api_key`: cheia personală
- `base_url`: adresa endpoint-ului Gemini compatibil cu formatul OpenAI
- `model`: modelul pe care îl vom folosi în apel

In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Missing GEMINI_API_KEY. Add it to your .env file.")

In [5]:
client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [6]:
from openai import OpenAI

client = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

MODEL = "gemini-2.5-flash"

print("Client configurat.")
print("Model selectat:", MODEL)

Client configurat.
Model selectat: gemini-2.5-flash


## 4. Primul apel minimal
Începem cu cel mai simplu caz: trimitem o singură întrebare către model și primim un răspuns.
Aici nu folosim încă `system message`, nu definim un rol special și nu construim o conversație.
Vrem doar să verificăm că:
- cheia API funcționează
- modelul răspunde
- înțelegem forma minimă a unui apel




In [7]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Say hello."}
    ]
)
print(response.choices[0].message.content)

Hello!


In [8]:
# raspunsul modelului 
response.choices[0].message.content

'Hello!'

In [9]:
response

ChatCompletion(id='p2gRarb9Ov2FkdUPqJncaA', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Hello!', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1779525800, model='gemini-2.5-flash', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=2, prompt_tokens=4, total_tokens=33, completion_tokens_details=None, prompt_tokens_details=None))

- ChatCompletion(...) = obiectul complet returnat de API. Nu este doar textul, ci răspunsul + metadata.
- id='JRvvaaXoMueynsEPusrdsQM' = identificator unic al apelului. Util pentru debugging/loguri.
- object='chat.completion' = tipul obiectului returnat. Confirmă că este un răspuns de tip chat completion.
- created=1777277734 = timestamp intern al momentului când a fost creat răspunsul.
- model='gemini-2.5-flash' = modelul care a generat răspunsul.
- choices=[...] = lista de răspunsuri generate. De obicei folosim primul răspuns: choices[0].
- index=0 = poziția răspunsului în lista choices. Aici este primul și singurul răspuns.
- message=ChatCompletionMessage(...) = mesajul generat de model.
- message.content='Hello!' = textul efectiv al răspunsului. Îl accesezi cu:
- response.choices[0].message.content
- message.role='assistant' = rolul mesajului generat. Modelul răspunde ca assistant.
- finish_reason='stop' = modelul s-a oprit normal. Nu a fost tăiat de limită.
- logprobs=None = nu ai cerut probabilități pentru tokeni.
- refusal=None = modelul nu a refuzat cererea.
- annotations=None = nu există adnotări suplimentare returnate.
- audio=None = nu există output audio.
- function_call=None = modelul nu a cerut apelarea unei funcții vechi-style.
- tool_calls=None = modelul nu a cerut folosirea unui tool.
- service_tier=None = nu apare un nivel special de serviciu raportat.
- system_fingerprint=None = providerul nu a returnat o amprentă internă a sistemului.
- usage=CompletionUsage(...) = informații despre tokeni consumați.
- prompt_tokens=4 = tokenii trimiși către model în cerere.
- completion_tokens=2 = tokenii generați de model în răspuns.
- total_tokens=31 = totalul raportat de provider. La Gemini prin compatibilitate OpenAI poate include overhead intern, deci nu trebuie tratat mereu ca simpla sumă prompt + completion.
- completion_tokens_details=None și prompt_tokens_details=None = nu există detalii suplimentare despre tokeni.

In [10]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": ""
            "You are a clear teacher explaining AI to sociology students."
        },
        {
            "role": "user",
            "content": "Explain what a large language model is in 3 simple sentences."
        }
    ],
    temperature=0.3
)

print(response.choices[0].message.content)

A Large Language Model (LLM) is a type of artificial intelligence designed to understand and generate human-like text. It learns by analyzing massive amounts of text from the internet, identifying patterns in language, grammar, and even some factual information. This extensive training allows it to perform various language tasks, from answering questions and writing stories to summarizing complex documents, by predicting the most probable next word.


## 5. Mai multe modele

In [11]:
models_to_test = [
    "gemini-2.5-flash",
    "gemini-2.5-flash-lite",
]

for model_name in models_to_test:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": "Explain what an LLM is in one simple sentence."}
        ]
    )
    print("=" * 60)
    print("MODEL:", model_name)
    print(response.choices[0].message.content)

MODEL: gemini-2.5-flash
An LLM is an artificial intelligence trained to understand and generate human language.
MODEL: gemini-2.5-flash-lite
An LLM is a computer program trained on a massive amount of text data that can understand and generate human-like language.


In [12]:
response

ChatCompletion(id='3mgRaqikIbafnsEPwsrHYQ', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='An LLM is a computer program trained on a massive amount of text data that can understand and generate human-like language.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1779525855, model='gemini-2.5-flash-lite', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=25, prompt_tokens=12, total_tokens=37, completion_tokens_details=None, prompt_tokens_details=None))

## 6. Exercițiu: rezumat neutru al unei știri
Vom da modelului un text scurt și îi cerem să producă un rezumat neutru.
Regulă:
- maximum 60 de cuvinte
- fără opinie personală
- fără exagerări
- fără informații care nu apar în text

In [14]:
news_text = """
Guvernul a anunțat un nou program de finanțare pentru modernizarea transportului public în orașele mari.
Programul include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital.
Reprezentanții ministerului spun că scopul este reducerea poluării și creșterea accesului la transport public.
Unele organizații civice au cerut criterii clare de alocare a fondurilor și transparență în implementare.
"""

prompt = f"""
Rezuma textul de mai jos în maximum 10 de cuvinte.
Rezumatul trebuie să fie neutru, clar și factual.
Nu adăuga informații care nu apar în text.

TEXT:
{news_text}
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You summarize public-interest texts in a neutral and factual way."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.1,
    
)

summary = response.choices[0].message.content

print(summary)

Guvernul anunță program modernizare transport public; organizații cer transparență.


## 7. Ce face `temperature`?
`temperature` controlează cât de variat este răspunsul.
- valori mici: răspunsuri mai stabile, mai previzibile
- valori mari: răspunsuri mai creative, dar uneori mai puțin precise
Pentru analiză socială, clasificare și rezumate factuale, începem de obicei cu valori mici: `0`, `0.2`, `0.3`.
Pentru generare creativă putem folosi valori mai mari.

In [15]:
test_prompt = """
Descrie în două propoziții ce este un model lingvistic mare.
Explică pentru studenți de sociologie, fără jargon tehnic.
"""

temperatures = [0.0, 0.7, 1.5]

for temp in temperatures:
    response = client.chat.completions.create(
        model="gemini-2.5-flash-lite",
        messages=[
            {
                "role": "system",
                "content": "You explain AI concepts clearly and briefly."
            },
            {
                "role": "user",
                "content": test_prompt
            }
        ],
        temperature=temp,
        max_tokens=120
    )

    print("=" * 80)
    print("TEMPERATURE:", temp)
    print(response.choices[0].message.content)

TEMPERATURE: 0.0
Un model lingvistic mare este ca un asistent super inteligent care a citit o cantitate uriașă de texte, de la cărți la articole de pe internet. Datorită acestei "lecturi" extinse, el poate înțelege și genera limbaj uman, ajutându-ne să scriem, să rezumăm informații sau chiar să purtăm conversații.
TEMPERATURE: 0.7
Un model lingvistic mare este ca un student care a citit milioane de cărți și articole, învățând cum să înțeleagă și să folosească limbajul uman în diverse moduri. Acest lucru îi permite să scrie texte coerente, să răspundă la întrebări și chiar să traducă, bazându-se pe tiparele și informațiile pe care le-a absorbit din tot materialul citit.
TEMPERATURE: 1.5
Un model lingvistic mare este ca o enciclopedie uriașă, antrenată pe o cantitate imensă de texte și conversații, capabilă să înțeleagă și să genereze limbaj uman într-un mod asemănător cu noi. El poate fi folosit pentru a analiza cum vorbim și scriem, pentru a crea noi forme de comunicare sau chiar pentr

## 8. Mini-interpretare
Completați după rulare:
1. Care răspuns a fost cel mai stabil?
2. Care răspuns a fost cel mai creativ?
3. Care răspuns este mai potrivit pentru cercetare academică?
4. Ce valoare de `temperature` ați folosi pentru rezumate neutre?

TODO:

1. Raspuncul cu temperature valoarea 0.0 este cel mai stabil și predictibil. A folosit sintagma "asistent super inteligent" dar e mai degraba directă și lipsită de ambiguitate.
2. Răspunsul cu TEMPERATURE: 1.5 este cel mai creativ, dar și mai puțin coerent. Folosește o sintagma mai complexă ("enciclopedie uriașă"), dar și adaugă idei suplimentare (analiza vorbirii și scrierii, "traducerea" ideilor complexe). Există un risc ușor de incoerență sau detalii mai puțin relevante, dar structura e mai elaborata
3. TEMPERATURE: 0.0
4. Între 0.1 și 0.3, ca sa permita o ușoară variație în exprimare

## 9. Funcție simplă pentru apeluri repetate


In [16]:
MODEL = "gemini-2.5-flash-lite"
def ask_model(
    user_message,
    system_message="You are a helpful assistant.",
    model=MODEL,
    temperature=0.3,
    max_tokens=300
):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response.choices[0].message.content

In [17]:
answer = ask_model(
    user_message="Explain the difference between an LLM and an AI agent in 4 bullet points.",
    system_message="You explain AI concepts to sociology students using simple language.",
    temperature=0.3,
    max_tokens=250
)

print(answer)

Here's the difference between an LLM and an AI agent, explained simply for sociology students:

*   **LLM (Large Language Model): The Super-Smart Talker.** Think of an LLM like a really, really well-read person who can understand and generate human-like text. It's amazing at answering questions, writing stories, summarizing things, and even translating languages, but it mostly just *talks* and *processes information*.

*   **AI Agent: The Doer with a Goal.** An AI agent is like a person (or a robot) who not only understands things but also has a specific *goal* and can *take actions* in the real or digital world to achieve it. It uses its understanding (sometimes from an LLM!) to plan and execute steps.

*   **LLM is a Tool, Agent is the Actor.** An LLM is a powerful tool that an agent might *use*. The agent is the one who decides *what* to do with that tool (and other tools) to accomplish its mission. For example, an agent might use an LLM to understand a user's request, then use othe

## 10. Salvăm outputul și parametrii


Trebuie să știm:
- ce model am folosit
- ce prompt am trimis
- ce parametri am setat
- ce răspuns am primit


In [20]:
import json
from datetime import datetime
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

experiment = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL,
    "temperature": 0.2,
    "max_tokens": 120,
    "task": "neutral_news_summary",
    "input_text": news_text,
    "output_text": summary
}

output_path = output_dir / "c1_first_summary.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(experiment, f, ensure_ascii=False, indent=2)

print("Experiment salvat în:", output_path)

Experiment salvat în: outputs\c1_first_summary.json


## 11. Verificăm fișierul salvat
Acum citim fișierul înapoi, ca să vedem că rezultatul a fost salvat corect.

In [21]:
with open(output_path, "r", encoding="utf-8") as f:
    saved_experiment = json.load(f)

saved_experiment

{'timestamp': '2026-05-23T11:52:54.042846',
 'model': 'gemini-2.5-flash-lite',
 'temperature': 0.2,
 'max_tokens': 120,
 'task': 'neutral_news_summary',
 'input_text': '\nGuvernul a anunțat un nou program de finanțare pentru modernizarea transportului public în orașele mari.\nProgramul include achiziția de autobuze electrice, modernizarea stațiilor și extinderea sistemelor de ticketing digital.\nReprezentanții ministerului spun că scopul este reducerea poluării și creșterea accesului la transport public.\nUnele organizații civice au cerut criterii clare de alocare a fondurilor și transparență în implementare.\n',
 'output_text': 'Guvernul anunță program modernizare transport public; organizații cer transparență.'}

## 12. Exercițiu scurt pentru studenți
Alegeți un text scurt, de 5–8 rânduri, dintr-o sursă publică.
Poate fi despre transport, sănătate, educație, mediu sau costul vieții.
Completați variabila de mai jos și generați un rezumat neutru.
Apoi salvați rezultatul.

In [23]:
student_text = """
TODO: Preşedintele SUA Donald Trump şi-a reafirmat într-un mod inedit interesul faţă de Groenlanda, 
printr-o imagine postată pe reţeaua sa, Truth Social, după ce, miercuri, trimisul special al Statelor Unite în Groenlanda, 
a declarat pentru AFP că Statele Unite trebuie să-şi consolideze prezenţa pe acest teritoriu autonom danez, 
iar joi groenlandezii au organizat un protest la adresa lui Trump în timp ce diplomaţii americani inaugurau un nou consulat.
"""

student_prompt = f"""
Rezuma textul de mai jos în maximum 20 de cuvinte.
Rezumatul trebuie să fie neutru, clar și factual.
Nu adăuga informații care nu apar în text.

TEXT:
{student_text}
"""

student_summary = ask_model(
    user_message=student_prompt,
    system_message="You summarize public-interest texts in a neutral and factual way.",
    temperature=0.2,
    max_tokens=120
)

print(student_summary)

Trump și-a reafirmat interesul față de Groenlanda printr-o postare, în timp ce SUA își consolidează prezența, iar groenlandezii protestează.


In [24]:
student_experiment = {
    "timestamp": datetime.now().isoformat(),
    "model": MODEL,
    "temperature": 0.2,
    "max_tokens": 120,
    "task": "student_neutral_summary",
    "input_text": student_text,
    "output_text": student_summary
}

student_output_path = output_dir / "c1_student_summary.json"

with open(student_output_path, "w", encoding="utf-8") as f:
    json.dump(student_experiment, f, ensure_ascii=False, indent=2)

print("Experiment salvat în:", student_output_path)

Experiment salvat în: outputs\c1_student_summary.json


## 13. Checklist pentru finalul Cursului 1
La finalul acestui notebook trebuie să aveți:
- API key funcțională
- `.env` local configurat
- primul apel către Gemini rulat cu succes
- un rezumat neutru generat
- experimentul cu `temperature` rulat
- output salvat în `outputs/`
- `.env` adăugat în `.gitignore`
Pentru repository:
- `README.md`
- `.env.example`
- `.gitignore`
- notebook-ul de Curs 1
- fără cheia API în GitHub

## 14. Ce luăm mai departe în Cursul 2
Acum știm să apelăm un model și să controlăm minim răspunsul.
În Cursul 2 vom compara modele:
- același prompt
- modele diferite
- criterii simple de evaluare: claritate, factualitate, neutralitate, viteză
Întrebarea următoare nu mai este „cum chemăm modelul?”, ci „ce model alegem pentru proiect?”.